# 02 — Evaluation

Loads a checkpoint, confirms no train/val leakage, computes val metrics,
and produces three figures:
- Score histogram (healthy vs. bleached softmax distributions)
- ROC curve with AUC
- Per-site accuracy bar chart

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
CKPT_PATH   = "/path/to/model_states/best.pt"
PROJECT_DIR = "/path/to/ct_classifier"   # contains ct_classifier/ source module
FIGURES_DIR = "figures"                  # output directory for saved plots

In [ ]:
import os, sys
import yaml
import numpy as np
import matplotlib.pyplot as plt
from sklearn import metrics
import torch
from torch.utils.data import DataLoader

# make ct_classifier source importable
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

from ct_classifier.dataset import BleachDataset
from ct_classifier.model import CustomResNet
from utils.evaluate import get_model_preds, eval_loss_acc, check_leakage

os.makedirs(FIGURES_DIR, exist_ok=True)

In [ ]:
# ── Load config + model ───────────────────────────────────────────────────────
cfg_path = os.path.join(os.path.dirname(CKPT_PATH), "config.yaml")
with open(cfg_path) as f:
    cfg = yaml.safe_load(f)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

model = CustomResNet(cfg["num_classes"], cfg["layers"])
state = torch.load(CKPT_PATH, map_location="cpu")
model.load_state_dict(state["model"])
model.to(device).eval()
print("Checkpoint loaded:", CKPT_PATH)

In [ ]:
# ── Build dataloaders ─────────────────────────────────────────────────────────
dl_kwargs = dict(
    batch_size=cfg.get("batch_size", 32),
    shuffle=False,
    num_workers=cfg.get("num_workers", 0),
    pin_memory=(device != "cpu"),
)
dl_train = DataLoader(BleachDataset(cfg, split="train", eval=True), **dl_kwargs)
dl_val   = DataLoader(BleachDataset(cfg, split="val",   eval=True), **dl_kwargs)

n_train, n_val = check_leakage(dl_train, dl_val)
print(f"Leakage check passed — train: {n_train} unique IDs, val: {n_val} unique IDs")

In [ ]:
# ── Metrics ───────────────────────────────────────────────────────────────────
val_loss, val_acc = eval_loss_acc(model, dl_val, device=device)
print(f"Val loss : {val_loss:.4f}")
print(f"Val acc  : {val_acc*100:.2f}%")

val_df = get_model_preds(model, dl_val, device=device)
val_df.head()

In [ ]:
# ── Figure 1: Score histogram ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))
for label, grp in val_df.groupby("str_labels"):
    ax.hist(grp["softmax_bleach_scores"], bins=20, range=(0, 1),
            alpha=0.6, label=label)
ax.set_xlabel("Softmax bleach score")
ax.set_ylabel("Count")
ax.set_title("Val set — score distribution")
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, "score_histogram.png"), dpi=150)
plt.show()

In [ ]:
# ── Figure 2: ROC curve ───────────────────────────────────────────────────────
fpr, tpr, _ = metrics.roc_curve(val_df["labels"], val_df["softmax_bleach_scores"])
auc = metrics.roc_auc_score(val_df["labels"], val_df["softmax_bleach_scores"])

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(fpr, tpr, label=f"AUC = {auc:.3f}")
ax.plot([0, 1], [0, 1], "k--", linewidth=0.8)
ax.set_xlabel("False positive rate")
ax.set_ylabel("True positive rate")
ax.set_title("Validation ROC")
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, "roc_curve.png"), dpi=150)
plt.show()

In [ ]:
# ── Figure 3: Per-site accuracy ───────────────────────────────────────────────
site_acc = (
    val_df.assign(correct=lambda d: (d["softmax_bleach_scores"] > 0.5).astype(int) == d["labels"])
    .groupby("site")["correct"].mean()
    .sort_values()
)

fig, ax = plt.subplots(figsize=(8, 4))
site_acc.plot(kind="barh", ax=ax, color="steelblue")
ax.axvline(0.5, color="red", linestyle="--", linewidth=0.8, label="chance")
ax.set_xlabel("Accuracy")
ax.set_title("Per-site accuracy (val set)")
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, "per_site_accuracy.png"), dpi=150)
plt.show()